# Stacking

This notebook implements a stacking ensemble learning technique (meta-learning) for fake news detection.

## Optimization Problem

MIN Z

$$
Z = - \frac{1}{N} \displaystyle\sum_{i=1}^{N} α_{c_i} [w_1 y_i log(F(x_i)) + w_0 (1 - y_i) log(1 - F(x_i))] + λ Ω(θ)
$$

<br>SUBJECT TO <br><br>

$C_1$: Stacking meta-learner

$$F(x) = g(f_1(x), f_2(x), ⋯ , f_M(x))$$

$C_2$: Feature Mapping

$$x_i = ϕ (title_i, text_i, category_i, dataset_i)$$

$C_3$: Binary Labels

$$y_i ∈ \{0, 1\}$$

$C_4$: Category Weight

$$α_{c} = \frac{1}{freq(c_i)}$$

$C_5$: Class Weight

$$w_1 = \frac{N}{2N_1}, \qquad w_0 = \frac{N}{2N_0}$$


## Environment Configuration

The following code cell contains most dependencies that will be used in this notebook.

In [37]:
%load_ext cudf.pandas

# Standard library imports
import warnings
import copy
import cudf
import cuml
import cupy as cp
import pandas as pd
import numpy as np
import scipy.sparse as sp

# Third party imports
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.metrics import roc_auc_score

from cuml.linear_model import LogisticRegression as cuLR



The cudf.pandas extension is already loaded. To reload it, use:
  %reload_ext cudf.pandas


NOTE: Some CPU-only components like `TfidfVectorizer`, `OneHotEncoder` and `CalibratedClassifierCV` have no cuML equivalents with full sparse-matrix + `sample_weight` support so they remain on the CPU.

In [38]:
# Utility: Safely convert cuDF Series or CuPy array to NumPy
def to_numpy(arr):
    if hasattr(arr, 'to_numpy'):
        return arr.to_numpy() # cuDF Series / cupy-backed cuDF
    if hasattr(arr, 'get'):
        return arr.get() # cupy.ndarray
    return np.asarray(arr)

## Colab Configuration

Run the code cell below to download data ingestion script from Github.

In [39]:
!wget https://raw.githubusercontent.com/3608Team10/COMP3608PROJECT/refs/heads/main/ingest_data.py

--2026-04-15 00:32:55--  https://raw.githubusercontent.com/3608Team10/COMP3608PROJECT/refs/heads/main/ingest_data.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10317 (10K) [text/plain]
Saving to: ‘ingest_data.py.4’

ingest_data.py.4    100%[===================>]  10.08K  --.-KB/s    in 0s      

2026-04-15 00:32:55 (137 MB/s) - ‘ingest_data.py.4’ saved [10317/10317]



## Data Ingestion

Load all fake news datasets using the shared 'ingest_data.py' script. <br>

The resulting dataframe `df` has five columns:
- `title` : title of news
- `text` : actual text content of news
- `category` : type of news
- `label` : 0 = fake, 1 = real
- `dataset` : source dataset of the news record

Duplicate and null text rows are removed during ingestion resulting in ~99K records across three datasets and eight original columns.

In [40]:
from ingest_data import load_datasets

pdf = load_datasets()
df = cudf.from_pandas(pdf)
del pdf

df.head()

------------------------------------------------------------
Fake News Dataset Ingestion
------------------------------------------------------------

Loading bhavikjikadara ...
Using Colab cache for faster access to the 'fake-news-detection' dataset.
[bhavik] Loaded 'true.csv': 21417 rows
Using Colab cache for faster access to the 'fake-news-detection' dataset.
[bhavik] Loaded 'fake.csv': 23481 rows

Loading mahdimashayekhi ...
Using Colab cache for faster access to the 'fake-news-detection-dataset' dataset.
[mahdi] Loaded 'fake_news_dataset.csv': 20000 rows

Loading shawkyelgendy ...
Using Colab cache for faster access to the 'fake-news-football' dataset.
[shawky] Loaded 'real.csv': 21871 rows
Using Colab cache for faster access to the 'fake-news-football' dataset.
[shawky] Loaded 'fake.csv': 20072 rows

Dropped 649 rows with empty/null text.
Dropped 6,650 duplicate rows.

------------------------------------------------------------
Fake News Dataset Summary
-------------------------

,title,text,label,category,dataset
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,1,Politics,bhavikjikadara
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,1,Politics,bhavikjikadara
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,1,Politics,bhavikjikadara
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,1,Politics,bhavikjikadara
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,1,Politics,bhavikjikadara


## Text Cleaning and Preprocessing Pipeline

Raw news text contains noise that degrades quality of TF-IDF. <br>

We concatenate `title` and `text` into a single `content` field, then apply a cleaning pipeline that:
- Converts all text to lower case
- Removes URLs
- Strips HTML entities and HTML tags
- Removes punctuation and digits
- Collapses whitespace

Note: Stemming/Lemmatisation is NOT applied at this stage. TF-IDF witha sublinear term-frequency scale handles term frequency inflation naturally <br>

The cleaned text is stored in the new column `content`.

In [41]:
def clean_text(series: cudf.Series) -> cudf.Series:
    series = series.str.lower()
    series = series.str.replace(r'https?://\S+|www\.\S+', ' ', regex=True) # Remove URLs
    series = series.str.replace(r'<[^>]+>', ' ', regex=True) # Strip HTML Tags
    series = series.str.replace(r'&[a-z]+;', ' ', regex=True) # Strip HTML Entities
    series = series.str.replace(r'[^a-z\s]', ' ', regex=True) # Keep letters only
    series = series.str.replace(r'\s+', ' ', regex=True) # Collapse whitespace
    series = series.str.strip()
    return series

df['content'] = clean_text(
    df['title'].fillna('') + ' ' + df['text'].fillna('')
)
df = df.reset_index(drop=True)

df.head()


,title,text,label,category,dataset,content
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,1,Politics,bhavikjikadara,as u s budget fight looms republicans flip the...
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,1,Politics,bhavikjikadara,u s military to accept transgender recruits on...
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,1,Politics,bhavikjikadara,senior u s republican senator let mr mueller d...
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,1,Politics,bhavikjikadara,fbi russia probe helped by australian diplomat...
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,1,Politics,bhavikjikadara,trump wants postal service to charge much more...


## TF-IDF Feature Extraction

$C_2$: $x_i = ϕ(title_i, text_i, category_i, dataset_i)$

Constraint $C_2$ maps raw inputs to a feature vector. The primary signal comes from TF-IDF on `content`. Use:

<pre>
sublinear_tf = True               - Dampens high-frequency terms
max_features = 100 000            - Vocabulary cap to control dimensionality
ngram_range = (1,2)               - unigrams + bigrams to capture short phrases
min_df = 3                        - ignore very rare terms
strip_accents / analyzer = 'word' - standard tokenization
</pre>

The vectorizer is fitted only on the training split to prevent data leakage. At this stage fit using the full dataframe to obtain the index. The fit is to be redone after train/test splitting.

In [42]:
TFIDF_PARAMS = {
    'sublinear_tf': True,
    'max_features': 100000,
    'ngram_range': (1, 2),
    'min_df': 3,
    'strip_accents': 'unicode',
    'analyzer': 'word',
    'token_pattern': r'\b[a-z]{2,}\b' # min 2 characters per word
}

tfidf = TfidfVectorizer(**TFIDF_PARAMS)

print("TF-IDF Vectorizer Configured")
print(f"    ngram_range  : {tfidf.ngram_range}")
print(f"    max_features : {tfidf.max_features:,}")
print(f"    sublinear_tf : {tfidf.sublinear_tf}")

TF-IDF Vectorizer Configured
    ngram_range  : (1, 2)
    max_features : 100,000
    sublinear_tf : True


## Categorical Encoding

The `category` and `dataset` columns are categorical. <br>
One-hot encode these columns and append them as sparse columns to the TF-IDF matrix so the full feature vector satisfies $C_2$. <br>
One-hot encoding is chosen over ordinal encoding because there is no inherent ordering among categories. <br>
We use `sparse_output=True` to keep memory usage manageable when concatenating with the TF-IDF matrix.

In [43]:
cat_encoder = OneHotEncoder(sparse_output=True, handle_unknown='ignore')

cat_encoder.fit(df[['category', 'dataset']].to_pandas())
cat_features_all = cat_encoder.transform(df[['category', 'dataset']].to_pandas())

print("Categorical Encoder Fitted")
print(f"    Input columns   : category, dataset")
print(f"    Output shape    : {cat_features_all.shape}")
print(f"    Category values : {cat_encoder.categories_[0].tolist()}")
print(f"    Dataset values  : {cat_encoder.categories_[1].tolist()}")

Categorical Encoder Fitted
    Input columns   : category, dataset
    Output shape    : (99542, 11)
    Category values : ['Business', 'Entertainment', 'Health', 'News', 'Politics', 'Science', 'Sports', 'Technology']
    Dataset values  : ['bhavikjikadara', 'mahdimashayekhi', 'shawkyelgendy']


## Sample Weight Computation

The objective function $Z$ uses two complementary weight terms:
1. **Category Weights** ($C_4$): Adjusts weights for over-represented categories (e.g. Sports = 44% of data) so that rarer categories (e.g. Science, Business) contributes proportionally.
2. **Class Weights** ($C_5$): Corrects for class imbalance between real and fake news.


**Category Weights** ($C_4$)

$$α_c = \frac{1}{freq(c_i)}$$

In [44]:
cat_freq = df['category'].value_counts(normalize=True).reset_index()
cat_freq.columns = ['category', 'freq']
cat_freq['alpha'] = 1.0 / cat_freq['freq']

category_weight_per_sample = (
    df[['category']].merge(
        cat_freq[['category', 'alpha']],
        on='category',
        how='left')['alpha']
)

print(f"\nCategory weight")
for row in cat_freq.to_pandas().itertuples():
    print(f"    {row.category:<20s}: alpha = {row.alpha:.4f}  (freq = {row.freq:.3f})")


Category weight
    Sports              : alpha = 2.2745  (freq = 0.440)
    Politics            : alpha = 4.6010  (freq = 0.217)
    News                : alpha = 5.0246  (freq = 0.199)
    Health              : alpha = 34.0664  (freq = 0.029)
    Entertainment       : alpha = 34.4555  (freq = 0.029)
    Technology          : alpha = 34.5392  (freq = 0.029)
    Business            : alpha = 34.9393  (freq = 0.029)
    Science             : alpha = 35.6909  (freq = 0.028)


**Class Weights** ($C_5$)

$$w_1 = \frac{N}{2N_1}, \qquad w_0 = \frac{N}{2N_0}$$

In [45]:
N = len(df)
label_counts = df['label'].value_counts()
N0 = int(label_counts[0])
N1 = int(label_counts[1])

# C5: w_i = N / (2 * N_i)
w0 = N / (2 * N0)
w1 = N / (2 * N1)

class_weight_map = {0: w0, 1: w1}
class_weight_per_sample = df['label'].map(class_weight_map)

print(f"Class weights")
print(f"    w0 (fake) : {w0:.4f}")
print(f"    w1 (real) : {w1:.4f}")

Class weights
    w0 (fake) : 1.0553
    w1 (real) : 0.9502


The final per-sample weight is the product $w_{y_i} ⋅ α_{c_i}$, which is passed as `sample_weight` to each base learner and the meta-learner.

In [46]:
sample_weights_cudf = class_weight_per_sample * category_weight_per_sample # Combine sample weights
sample_weights_cudf = sample_weights_cudf / sample_weights_cudf.mean() # Normalise so weights sum to N
sample_weights = to_numpy(sample_weights_cudf) # Convert to numpy for sklean/cuML

df['sample_weight'] = sample_weights_cudf

print(f"\nSample weights stats")
print(f"    min  : {sample_weights.min():.4f}")
print(f"    max  : {sample_weights.max():.4f}")
print(f"    mean : {sample_weights.mean():.4f}")


Sample weights stats
    min  : 0.2697
    max  : 4.7008
    mean : 1.0000


## Stratified Train/Test Split

Train data will comprise of 80% of the data. <br>
Test data will comprise of 20% of the data. Test data is never touched during model training or OOF generation. <br>

Stratification is performed jointly on `label` and `category` to ensure both class and category distributions are preserved in both splits. This prevents distribution shift between training and evaluation. <br>

After splitting, the TF-IDF vectorizer is fitted only on training data and used to transform both splits preventing any leakage of test vocabulary into feature weights.

In [47]:
# Create a joint stratification key: label + category
strat_key = (df['label'].astype(str) + '_' + df['category']).to_pandas()

idx = np.arange(len(df))
idx_train, idx_test = train_test_split(
    idx,
    test_size=0.20,
    random_state=42,
    stratify=strat_key
)

df_train = df.iloc[idx_train].reset_index(drop=True)
df_test = df.iloc[idx_test].reset_index(drop=True)
sw_train = sample_weights[idx_train]
sw_test = sample_weights[idx_test]

# Fit TF-IDF on train only, transform both splits
X_train_tfidf = tfidf.fit_transform(df_train['content'].to_pandas())
X_test_tfidf = tfidf.transform(df_test['content'].to_pandas())

# Categorical features - encoder already fitted on full data
X_train_cat = cat_encoder.transform(df_train[['category', 'dataset']].to_pandas())
X_test_cat = cat_encoder.transform(df_test[['category', 'dataset']].to_pandas())

# Full feature matrices
X_train = sp.hstack([X_train_tfidf, X_train_cat], format='csr')
X_test = sp.hstack([X_test_tfidf, X_test_cat], format='csr')

Y_train = to_numpy(df_train['label'])
Y_test = to_numpy(df_test['label'])

print(f"Train size : {len(df_train):>7,}")
print(f"Test size  : {len(df_test):>6,}")
print(f"X_train    : {X_train.shape}")
print(f"X_test     : {X_test.shape}")
print(f"\nTrain label distribution")
print(f"    Fake: {(Y_train==0).sum():,}")
print(f"    Real: {(Y_train==1).sum():,}")
print(f"\nTest label distribution")
print(f"    Fake: {(Y_test==0).sum():,}")
print(f"    Real: {(Y_test==1).sum():,}")

Train size :  79,633
Test size  : 19,909
X_train    : (79633, 100011)
X_test     : (19909, 100011)

Train label distribution
    Fake: 37,729
    Real: 41,904

Test label distribution
    Fake: 9,432
    Real: 10,477


## K-Fold OOF Framework

The stacking meta-learner $g$ is trained on Out-of-Fold (OOF) predictions from the base models. <br>
How it works:
1. The training set is split into $K$ Folds
2. Each base model is trained on $K - 1$ Folds and predicts on the $K^{th}$ fold
3. After $K$ rotations, every training sample has exactly one OOF prediction made by a model that never saw that sample during training

This ensures the meta-learner is trained on unbiased base-model outputs. We use Stratified K-Fold ($K = 5$) so each fold preserves class and category balance. <br>

A helper function `generate_oof` runs the stratified K-fold OOF loop for a fitted-style estimator <br>

Parameters <br>
- estimator : sklearn estimator with predict_proba
- X : sparse feature matrix (train split)
- Y : label array
- sample_weight : per-sample weights
- cv : StratifiedKFold splitter

Returns <br>
- oof_proba : ndarray of shape (n_samples,) - $P(real | x)$ for each OOF sample
- fold_models : list of fitted clones

In [48]:
warnings.filterwarnings('ignore')

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)


def clone_est(estimator):
    try:
        return clone(estimator)
    except Exception:
        return copy.deepcopy(estimator)


def generate_oof(estimator, X, Y, sample_weight, cv):
    oof_proba = np.zeros(len(Y))
    fold_models = []

    for fold, (tr_idx, val_idx) in enumerate(cv.split(X, Y), 1):
        X_tr, X_val = X[tr_idx], X[val_idx]
        Y_tr = Y[tr_idx]
        sw_tr = sample_weight[tr_idx]

        model = clone_est(estimator)
        model.fit(X_tr, Y_tr, sample_weight=sw_tr)

        proba = to_numpy(model.predict_proba(X_val))[:, 1]
        oof_proba[val_idx] = proba
        fold_models.append(model)
        print(f"    Fold {fold}/{N_SPLITS} done", end="\r")

    print()
    return oof_proba, fold_models


print(f"OOF Framework ready: {N_SPLITS}-fold StratifiedKFold")

OOF Framework ready: 5-fold StratifiedKFold


## Weak Learner: Logistic Regression

In [50]:
# Tune regularization and penalty type

lr_C_values = [0.01, 0.1, 1.0, 5.0]
lr_penalty_values = ['l1', 'l2']

best_lr_auc = -1
best_lr_params = {}

inner_cv_lr = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)

for C_val in lr_C_values:
    for penalty_val in lr_penalty_values:
        cv_aucs = []
        for tr_idx, val_idx in inner_cv_lr.split(X_train, Y_train):
            lr_t = cuLR(
                C=C_val,
                penalty=penalty_val,
                solver='qn',
                max_iter=1000
            )
            lr_t.fit(
                X_train[tr_idx],
                Y_train[tr_idx],
                sample_weight=sw_train[tr_idx]
            )
            proba = to_numpy(lr_t.predict_proba(X_train[val_idx]))[:, 1]
            cv_aucs.append(roc_auc_score(Y_train[val_idx], proba))

        mean_auc = np.mean(cv_aucs)
        print(f'    C = {C_val} | penalty = {penalty_val} | AUC = {mean_auc:.4f}')
        if mean_auc > best_lr_auc:
            best_lr_auc = mean_auc
            best_lr_params = {
                'C': C_val,
                'penalty': penalty_val
            }

print(f'\nBest LR params : {best_lr_params}')
print(f'Best CV AUC : {best_lr_auc:.4f}')

lr_best = cuLR(
    C = best_lr_params['C'],
    penalty = best_lr_params['penalty'],
    solver = 'qn',
    max_iter = 1000
)

    C = 0.01 | penalty = l1 | AUC = 0.5297
    C = 0.01 | penalty = l2 | AUC = 0.8892
    C = 0.1 | penalty = l1 | AUC = 0.9360
    C = 0.1 | penalty = l2 | AUC = 0.9442
    C = 1.0 | penalty = l1 | AUC = 0.9550
    C = 1.0 | penalty = l2 | AUC = 0.9569
[2026-04-15 00:42:14.999] [CUML] [warning] QWL-QN: max iterations reached
[2026-04-15 00:42:14.999] [CUML] [warning] Maximum iterations reached before solver is converged. To increase model accuracy you can increase the number of iterations (max_iter) or improve the scaling of the input data.
[2026-04-15 00:42:18.173] [CUML] [warning] QWL-QN: max iterations reached
[2026-04-15 00:42:18.173] [CUML] [warning] Maximum iterations reached before solver is converged. To increase model accuracy you can increase the number of iterations (max_iter) or improve the scaling of the input data.
[2026-04-15 00:42:21.724] [CUML] [warning] QWL-QN: max iterations reached
[2026-04-15 00:42:21.724] [CUML] [warning] Maximum iterations reached before solver 